# Sidebar Links Validation Tool

This notebook validates that all sidebar links point to existing pages in the `/templates` directory, and creates any missing pages to ensure the entire navigation structure works properly.

In [ ]:
# Import Required Libraries
import os
import json
import re
from pathlib import Path
import datetime

## Load Sidebar Links

We'll parse the sidebar configuration file to extract all links that should point to pages in the templates directory.

In [ ]:
# Define paths
base_dir = Path("d:/Projects/impressioncore")
templates_dir = base_dir / "templates"
sidebar_config_path = base_dir / "config" / "sidebar.json"

# Check if sidebar configuration exists
if not sidebar_config_path.exists():
    sidebar_config_path = base_dir / "sidebar.json"  # Try alternative location
    if not sidebar_config_path.exists():
        print(f"Could not find sidebar configuration at {sidebar_config_path}")
        # Look for potential sidebar config files
        potential_files = list(base_dir.glob("**/sidebar*.json"))
        if potential_files:
            print(f"Found potential sidebar configuration files:")
            for file in potential_files:
                print(f"- {file}")
            sidebar_config_path = potential_files[0]
            print(f"Using {sidebar_config_path}")
        else:
            raise FileNotFoundError("No sidebar configuration file found")

# Load sidebar configuration
with open(sidebar_config_path, 'r') as f:
    try:
        sidebar_config = json.load(f)
        print(f"Successfully loaded sidebar configuration from {sidebar_config_path}")
    except json.JSONDecodeError:
        print(f"Error parsing JSON from {sidebar_config_path}")
        raise

## Extract Links from Sidebar Configuration

We need to traverse the sidebar configuration structure to extract all links. Sidebar configurations can be nested, so we'll use a recursive approach.

In [ ]:
def extract_links(items):
    """Recursively extract all links from sidebar items"""
    links = []
    
    if not isinstance(items, list):
        return links
        
    for item in items:
        if isinstance(item, dict):
            # If item has a link, add it
            if 'link' in item and item['link']:
                links.append(item['link'])
                
            # If item has children, process them recursively
            if 'children' in item and item['children']:
                links.extend(extract_links(item['children']))
    
    return links

# Extract all links from the sidebar configuration
all_sidebar_links = extract_links(sidebar_config.get('items', []))

# Clean up links - remove any fragment identifiers and query parameters
def clean_link(link):
    # Remove fragment identifiers and query parameters
    link = link.split('#')[0].split('?')[0]
    # Remove leading slash if present
    return link.lstrip('/')

# Clean and normalize the links
sidebar_links = [clean_link(link) for link in all_sidebar_links if link]
print(f"Found {len(sidebar_links)} links in the sidebar configuration")
print("First few links:", sidebar_links[:5])

## Check for Missing Pages

We'll check if each link in the sidebar configuration has a corresponding file in the templates directory.

In [ ]:
# Function to check if a page exists
def page_exists(link):
    # Add .html extension if not present
    if not link.endswith('.html'):
        link_path = link + '.html'
    else:
        link_path = link
        
    # Check if the file exists in the templates directory
    file_path = templates_dir / link_path
    return file_path.exists(), file_path

# Check each link and categorize as existing or missing
existing_links = []
missing_links = []

for link in sidebar_links:
    exists, file_path = page_exists(link)
    if exists:
        existing_links.append((link, file_path))
    else:
        missing_links.append((link, file_path))

print(f"\nResults:")
print(f"- {len(existing_links)} pages already exist")
print(f"- {len(missing_links)} pages are missing")

# Show some of the missing links
if missing_links:
    print("\nFirst few missing pages:")
    for i, (link, path) in enumerate(missing_links[:5]):
        print(f"{i+1}. {link} -> {path}")

## Create Missing Pages

For each missing page, we'll create a new file in the templates directory with a basic template structure.

In [ ]:
def create_template_page(file_path, title):
    """Create a basic template page"""
    # Ensure the parent directory exists
    file_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Generate a title from the link
    page_title = title.replace('-', ' ').replace('_', ' ').title()
    
    # Basic template content
    template_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{page_title}</title>
    <link rel="stylesheet" href="/static/css/style.css">
</head>
<body>
    <div class="container">
        <header>
            <h1>{page_title}</h1>
        </header>
        
        <main>
            <section>
                <h2>About This Page</h2>
                <p>This page was automatically generated on {datetime.datetime.now().strftime('%Y-%m-%d')} to support the site navigation structure.</p>
                <p>Replace this content with your actual page content.</p>
            </section>
        </main>
        
        <footer>
            <p>&copy; {datetime.datetime.now().year} ImpressionCore Project</p>
        </footer>
    </div>
    
    <script src="/static/js/script.js"></script>
</body>
</html>
"""
    
    # Write content to the file
    with open(file_path, 'w') as f:
        f.write(template_content)
    
    return True

In [ ]:
# Create missing pages
created_pages = []
failed_creations = []

for link, file_path in missing_links:
    title = os.path.basename(link)
    try:
        if create_template_page(file_path, title):
            created_pages.append((link, file_path))
        else:
            failed_creations.append((link, file_path))
    except Exception as e:
        print(f"Error creating {file_path}: {str(e)}")
        failed_creations.append((link, file_path))

print(f"\nCreated {len(created_pages)} missing pages")
if failed_creations:
    print(f"Failed to create {len(failed_creations)} pages")

## Update Sidebar Links

Ensure that all sidebar links point to the correct files in the `/templates` directory by validating the path structure.

In [ ]:
def validate_link_format(link):
    """Check if a link follows the proper format and suggest corrections if needed"""
    original_link = link
    
    # Ensure link has no leading slash (relative to templates directory)
    if link.startswith('/'):
        link = link[1:]
    
    # Ensure link has .html extension if it's not pointing to a directory
    if not link.endswith('/') and not link.endswith('.html'):
        link += '.html'
    
    return link, (original_link != link)

# Validate and suggest corrections for all sidebar links
corrections_needed = []

for i, link in enumerate(sidebar_links):
    corrected_link, needs_correction = validate_link_format(link)
    if needs_correction:
        corrections_needed.append((link, corrected_link))

print(f"\n{len(corrections_needed)} links need format corrections")
if corrections_needed:
    print("\nSuggested link corrections:")
    for i, (original, corrected) in enumerate(corrections_needed[:10]):
        print(f"{i+1}. {original} -> {corrected}")
    
    if len(corrections_needed) > 10:
        print(f"... and {len(corrections_needed) - 10} more")

## Generate Report of Link Status

Generate a comprehensive report listing the status of all sidebar links: which links are complete and working, which links were missing and created, and which links need format corrections.

In [ ]:
# Generate report
report = {
    "total_links": len(sidebar_links),
    "existing_pages": [{
        "link": link,
        "path": str(path)
    } for link, path in existing_links],
    "created_pages": [{
        "link": link,
        "path": str(path)
    } for link, path in created_pages],
    "failed_creations": [{
        "link": link,
        "path": str(path)
    } for link, path in failed_creations],
    "format_corrections": [{
        "original": original,
        "corrected": corrected
    } for original, corrected in corrections_needed]
}

# Save report to file
report_path = base_dir / "tools" / "sidebar_links_report.json"
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f"\nFull report saved to {report_path}")

## Summary

This notebook has checked all sidebar links against the templates directory, created missing pages, and identified format issues in the sidebar configuration.

### Results:
- Total links analyzed: {len(sidebar_links)}
- Existing pages: {len(existing_links)}
- Missing pages created: {len(created_pages)}
- Failed creations: {len(failed_creations)}
- Links needing format corrections: {len(corrections_needed)}

To ensure a fully functional navigation system:
1. Review the created pages and add appropriate content
2. Fix any format issues in the sidebar configuration
3. Address any failed page creations

In [ ]:
# Print summary statistics
print("\n=== SUMMARY ===")
print(f"Total links analyzed: {len(sidebar_links)}")
print(f"Existing pages: {len(existing_links)}")
print(f"Missing pages created: {len(created_pages)}")
print(f"Failed creations: {len(failed_creations)}")
print(f"Links needing format corrections: {len(corrections_needed)}")

print("\nNavigation system health: ", end="")
if len(failed_creations) == 0 and len(corrections_needed) == 0:
    print("✅ PERFECT - All links now have pages and are correctly formatted")
elif len(failed_creations) == 0:
    print("⚠️ GOOD - All pages exist but some links need format corrections")
else:
    print("❌ NEEDS ATTENTION - Some pages couldn't be created")